# High Frequency Words Export

This notebook extracts all words from `output/words_database.sqlite` whose total frequency is above a configurable threshold and whose POS category is limited to verbs, nouns, or proper nouns (configurable). The result is written to a new SQLite database `output/high_frequency_words.sqlite` containing:

- `words` table (subset of original)
- `word_frequencies` yearly frequency rows for just those words

Adjust the parameters in the next cell and re-run the execution cell to regenerate the export.

In [ ]:
# Parameters
MIN_FREQUENCY = 1000  # Change threshold as needed
ALLOWED_CATEGORIES = ("verb", "noun", "proper_noun")  # Edit categories if desired

SOURCE_DB = 'output/words_database.sqlite'
TARGET_DB = 'output/high_frequency_words.sqlite'

import os, sqlite3
from datetime import datetime
from pathlib import Path

print(f'Source DB: {SOURCE_DB}')
print(f'Target DB: {TARGET_DB}')
print(f'Min frequency: {MIN_FREQUENCY}')
print(f'Categories: {ALLOWED_CATEGORIES}')

In [ ]:
def prepare_target(conn: sqlite3.Connection):
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS word_frequencies")
    cur.execute("DROP TABLE IF EXISTS words")
    cur.execute(
        """
        CREATE TABLE words (
            id INTEGER PRIMARY KEY,
            word TEXT NOT NULL,
            lemma TEXT NOT NULL,
            pos_tag TEXT NOT NULL,
            pos_category TEXT NOT NULL,
            total_frequency INTEGER DEFAULT 0,
            first_seen TEXT,
            last_seen TEXT,
            created_at TEXT DEFAULT (datetime('now'))
        )
        """
    )
    cur.execute(
        """
        CREATE TABLE word_frequencies (
            id INTEGER PRIMARY KEY,
            word_id INTEGER NOT NULL,
            year INTEGER NOT NULL,
            frequency INTEGER DEFAULT 0
        )
        """
    )
    cur.execute("CREATE INDEX idx_words_pos_cat_freq ON words(pos_category, total_frequency)")
    cur.execute("CREATE INDEX idx_word_freq_word_id ON word_frequencies(word_id)")
    conn.commit()

In [ ]:
def export(min_frequency=MIN_FREQUENCY, categories=ALLOWED_CATEGORIES, source_db=SOURCE_DB, target_db=TARGET_DB):
    if not os.path.exists(source_db):
        raise FileNotFoundError(f'Source database not found: {source_db}')
    if os.path.exists(target_db):
        os.remove(target_db)
    with sqlite3.connect(source_db) as src, sqlite3.connect(target_db) as dst:
        src.row_factory = sqlite3.Row
        prepare_target(dst)
        src_cur = src.cursor()
        dst_cur = dst.cursor()
        placeholders = ','.join(['?'] * len(categories))
        src_cur.execute(
            f'''
            SELECT * FROM words
            WHERE total_frequency > ?
              AND pos_category IN ({placeholders})
            ''',
            (min_frequency, *categories),
        )
        words = src_cur.fetchall()
        if not words:
            print('No words matched criteria.')
            return {'words': 0, 'freq_rows': 0}
        dst_cur.executemany(
            '''
            INSERT INTO words (id, word, lemma, pos_tag, pos_category, total_frequency, first_seen, last_seen, created_at)
            VALUES (:id, :word, :lemma, :pos_tag, :pos_category, :total_frequency, :first_seen, :last_seen, :created_at)
            ''',
            words,
        )
        word_ids = [row['id'] for row in words]
        CHUNK = 800
        total_freq_rows = 0
        src_cur2 = src.cursor()
        for i in range(0, len(word_ids), CHUNK):
            chunk = word_ids[i:i+CHUNK]
            ph = ','.join(['?'] * len(chunk))
            src_cur2.execute(f'SELECT * FROM word_frequencies WHERE word_id IN ({ph})', chunk)
            freq_rows = src_cur2.fetchall()
            if freq_rows:
                dst_cur.executemany(
                    '''
                    INSERT INTO word_frequencies (id, word_id, year, frequency)
                    VALUES (:id, :word_id, :year, :frequency)
                    ''',
                    freq_rows,
                )
                total_freq_rows += len(freq_rows)
        dst.commit()
        summary = {
            'words': len(words),
            'freq_rows': total_freq_rows,
            'target_db': target_db,
            'criteria': {'min_frequency': min_frequency, 'categories': categories},
            'timestamp_utc': datetime.utcnow().isoformat() + 'Z'
        }
        return summary

In [ ]:
# Run the export
summary = export()
if summary and summary['words'] > 0:
    print('Export complete:')
    for k, v in summary.items():
        if k != 'criteria':
            print(f'  {k}: {v}')
    print('  criteria:', summary['criteria'])
else:
    print('Nothing exported.')

In [ ]:
# Summarize by category from the new database
if Path(TARGET_DB).exists():
    with sqlite3.connect(TARGET_DB) as conn:
        cur = conn.cursor()
        cur.execute(
            '''SELECT pos_category, COUNT(*) as cnt,
                      MIN(total_frequency), MAX(total_frequency)
               FROM words
               GROUP BY pos_category
               ORDER BY cnt DESC'''
        )
        rows = cur.fetchall()
    print('Category summary:')
    for cat, cnt, minf, maxf in rows:
        print(f'  {cat}: {cnt} (min {minf}, max {maxf})')
else:
    print('Target DB not found yet.')